### Notebook for developing and testing a new MAF stacker class that will calculate an estimate of the DCR second moment additive bias for an "average" galaxy 

In [6]:
# from rubin_sim import maf
# from rubin_sim.data import get_data_dir

try:
    from rubin_sim.data import get_baseline
except ImportError:
    from rubin_scheduler.data import get_baseline
# import rubin_sim
# from rubin_sim import data
from rubin_sim.data import get_baseline
# import healpy as hp
import numpy as np
# import sqlite3
# from astropy.coordinates import SkyCoord
# import astropy.units as u
# import datashader as ds
# import matplotlib.pyplot as plt
# import matplotlib as mpl
# from datashader.mpl_ext import dsshow
# import pandas as pd
import galsim
# import matplotlib.lines as mlines
# import os

from scipy.interpolate import interp1d

from astropy.modeling.models import BlackBody
from astropy import units as u


def BB(wave, temp, scale):
    temp = temp * u.K
    wave = wave * u.nm
    scale = scale * u.erg / (u.cm**2 * u.s * u.AA * u.sr)

    bb = BlackBody(temperature = temp, scale = scale)
    return bb(wave).value

def apply_filter(wvl, data, band = 'g', rm_leakage = True):
    '''
    wvl - a list of the wavelengths in angstroms
    data - a N x M list of SEDs, N = number of SEDs, M = len of wvl
    rm_leakage - whether to set the filter throughput to 0 for wavelengths where the throughput is less than 0.001 (removing filter leakage)
    returns data with the band filter applied to each SED
    '''
    
    filter_file = f'filter_files/total_{band}.dat'
    filter_band = np.loadtxt(filter_file).T #filter_band[0] -> wavelengths, filter_band[1] -> filter pass fraction

    # #remove filter leakage: set lower bound of filter to 0.001 throughput
    if rm_leakage:
        leakage_mask = filter_band[1] < 0.001
        filter_band[1][leakage_mask] = np.zeros(np.sum(leakage_mask))
    
    func = interp1d(filter_band[0], filter_band[1])
    SED_filter = func(wvl)
    
    return data * SED_filter

### Chromatic refraction angle R of a light ray entering the atmosphere at a zenith angle $z_a$ is 
### $$\bold{R} = \bold{R}_{45} \tan{z_a},$$
### where $R_{45}$ is a constant depending on the wavelength of the light. 

Within a filter band, the bluer end of the band will be refracted more than the red band, this leads to the increase of the second moment of the PSF. 

DCR is an additive bias on the PSF second moment towards zenith, and is proportional to $R^2$, therefore 
$$ I_{DCR} = I_{DCR_{45}} \tan^2{z_a}$$

Below, we calculate the constants $I_{DCR_{45}}$ for each band (colloqiually called "widths"), assuming a flat SED in wavelength space. These values will be used in the MAF stacker.  

In [7]:
#Calculate the first and second moment shifts from DCR for flat SEDs ("average" Galaxy SED assumed to be flat)

filter_labels = ['u', 'g', 'r', 'i', 'z', 'y']

#get the filtered flat SED (in this case, these are exactly the response curves)
wavelengths_data = np.linspace(301, 1150, 10000)
filtered_flats = {}
for band in filter_labels:
    filtered_flats[band] = apply_filter(wavelengths_data , np.ones(10000), band, rm_leakage = True)


#For further investigation: generate a SED for a 5000K blackbody, the assumed average SED for a standard calibration star
# filtered_calib = {}
# for band in filter_labels:
#     filtered_calib[band] = apply_filter(wavelengths_data , BB(wavelengths_data, 5000, 1), band, rm_leakage = True)
    


def get_shifts(zeniths, wavelengths_data, filtered_data):
    '''
    zeniths - list or array of zenith angles in degrees

    returns shifts (mean of refraction dist, 1st moment) and widths (variance of refraction dist, 2nd moment) dictionaries for each filter band
    shifts - a dictionary with keys = filter bands, values = list of means for each zenith angle
    widths - a dictionary with keys = filter bands, values = list of variances for each zenith angle

    Calculates the distribution of refraction angles (dN/dR) by multiplying the SED (dN/dwvl) by the derivative of refraction with respect to wavelength (dwvl/dR)
    '''

    dwvl = 0.1 #wavelength step
    wavelengths = np.arange(290, 1160, dwvl)
    
    #Matthew's atmospheric conditions
    pressure=70.0,  # kPa
    temperature=293.15,  # K
    H2O_pressure=0.0,
    
    #Pat and Josh's paper atmospheric conditions 
    pressure=69.328
    temperature=293.15
    H2O_pressure=1.067

    #setup storage for results
    widths = {}
    for band in filter_labels:
        widths[band] = []

    shifts = {}
    for band in filter_labels:
        shifts[band] = []

        
    for zenith in zeniths:
        
        #Get dR/dwavelength function using the detailed wavelengths list 
        refraction = galsim.dcr.get_refraction(
                wavelengths,
                zenith_angle=zenith * galsim.degrees,
                pressure=pressure,  # kPa
                temperature=temperature,  # K
                H2O_pressure=H2O_pressure,  # kPa
                ) * 180 * 3600 / np.pi #convert from radians to arcsec
        
        dR_dwvl = (refraction[1:] - refraction[0:-1])/dwvl
        wavelengths_midpoints = wavelengths[0:-1] + dwvl/2
        
        dRdwvl_func = interp1d(wavelengths_midpoints, dR_dwvl)
        
        
        #Map each wavelength in our SED to a refraction angle 
        refraction_wave = galsim.dcr.get_refraction(
                wavelengths_data,
                zenith_angle=zenith * galsim.degrees,
                pressure=pressure,  # kPa
                temperature=temperature,  # K
                H2O_pressure=H2O_pressure,  # kPa
                ) * 180 * 3600 / np.pi #convert from radians to arcsec
        
        
        filtered_refracted_data = {} #stores dNdR for each band 
        
        
        for band, filtered_flat in filtered_data.items():
            #data is proportional to photons/sec/cm^2/nm so is proportional to dN/dwvl
            #get dN/dR with (dN/dwvl) / (dR/dwvl)
            dNdR = np.abs(filtered_flat / dRdwvl_func(wavelengths_data))
            filtered_refracted_data[band] = dNdR

            #calculate the mean and variance refraction angle using weighted sum method 
            mean = np.sum(dNdR * np.array(refraction_wave).T) / np.sum(dNdR)
            variance = np.sum(dNdR * (np.array(refraction_wave).T - mean)**2) / np.sum(dNdR)

            widths[band].append(variance)
            shifts[band].append(mean) #shift from the u band mean refraction angle
            
        

    return shifts, widths

        

zeniths = np.array([45])

shifts, widths = get_shifts(zeniths, wavelengths_data, filtered_flats)
# shifts_calib, widths_calib = get_shifts(zeniths, wavelengths_data, filtered_calib)

for band in filter_labels:
    print(band, 'first moment (shift): ', np.round(shifts[band][0], 4), 'arcsec   second moment (width): ', np.round(widths[band][0], 4), 'arcsec^2') 




u first moment (shift):  39.3492 arcsec   second moment (width):  0.024 arcsec^2
g first moment (shift):  38.638 arcsec   second moment (width):  0.0308 arcsec^2
r first moment (shift):  38.2577 arcsec   second moment (width):  0.0058 arcsec^2
i first moment (shift):  38.0773 arcsec   second moment (width):  0.0015 arcsec^2
z first moment (shift):  37.9863 arcsec   second moment (width):  0.0004 arcsec^2
y first moment (shift):  37.9262 arcsec   second moment (width):  0.0003 arcsec^2


### The second moment magnitudes for zenith angle = 45 degrees (printed in the cell above) will be used in the stacker as the dcr2_magnitudes

In [ ]:


# test DCR2 stacker
from rubin_sim.maf.stackers import BaseStacker
from rubin_sim.maf.stackers import ParallacticAngleStacker
from rubin_sim.maf.stackers import ZenithDistStacker

#Remove any existing Dcr2Stacker from the registry to avoid conflicts
bad_keys = [k for k in BaseStacker.registry if k.endswith(f".Dcr2Stacker")]
for k in bad_keys:
    BaseStacker.registry.pop(k, None)  # in case already registered
    

class Dcr2Stacker(BaseStacker):
    """
    
    NEEDS UPDATE FOR SECOND MOMENT CALCULATION
    Adapted from the original DcrStacker in rubin_sim.maf.stackers.dcrStacker
    
    Add columns representing the expected RA/Dec image spread contribution (variance) expected for
    an object due to differential chromatic refraction across the band, per visit.

    For DCR calculation, we also need zenithDistance, HA, and PA -- but these
    will be explicitly handled within this stacker so that setup is consistent
    and they run in order. If those values have already been calculated
    elsewhere, they will not be overwritten.

    Parameters
    ----------
    filter_col : `str`, optional
        The name of the column with filter names. Default 'filter'.
    altCol : `str`, optional
        Name of the column with altitude info. Default 'altitude'.
    ra_col : `str`, optional
        Name of the column with RA. Default 'fieldRA'.
    dec_col : `str`, optional
        Name of the column with Dec. Default 'fieldDec'.
    lstCol : `str`, optional
        Name of the column with local sidereal time. Default
        'observationStartLST'.
    site : `str` or `rubin_scheduler.utils.Site`, optional
        Name of the observory or a rubin_scheduler.utils.Site object.
        Default 'LSST'.
    mjdCol : `str`, optional
        Name of column with modified julian date.
        Default 'observationStartMJD'
    dcr2_magnitudes : dict, optional
        Magnitude of the DCR variance for each filter at an
        altitude/zenith distance of 45 degrees.
        Defaults u=0.0240, g=0.0308, r=0.0058, i=0.0015, z=0.0004, y=0.0003
        (all values should be in arcseconds^2).
    ra_and_dec : bool, optional
        Whether to break the DCR offset into RA and Dec components (True) 
        or just add a single ellipticity value in the direction toward zenith in a column 'dcr_var' (False, default). 
        If True, adds three columns: 'ra_dcr_var', 'dec_dcr_var', and 'ra_dec_dcr_cov'
        which can be thought of as shape moments. 
    

    Returns
    -------
    data : `numpy.array`
        Returns array with additional columns 'ra_dcr_amp' and 'dec_dcr_amp'
        with the DCR offsets for each observation.  Also runs ZenithDistStacker
        and ParallacticAngleStacker.
    """

    cols_added = ["dcr_var", "ra_dcr_var", "dec_dcr_var", "ra_dec_dcr_cov"]  # zenithDist, HA, PA

    def __init__(
        self,
        filter_col="band",
        alt_col="altitude",
        degrees=True,
        ra_col="fieldRA",
        dec_col="fieldDec",
        lst_col="observationStartLST",
        site="LSST",
        mjd_col="observationStartMJD",
        dcr2_magnitudes=None,
        ra_and_dec=False, #If False, just adds a single ellipticity value column implied to be in the direction toward zenith
                         #If True, adds two columns with the ellipticity broken into ra and dec components
    ):
        self.units = ["arcsec", "arcsec"]

        #Calculated using a flat SED in wavelength space, units are arcseconds squared
        if dcr2_magnitudes is None:
            self.dcr2_magnitudes = {
                "u": 0.02404978, #FILL THIS IN WITH THE VALUES CALCULATED FOR A FLAT SED 
                "g": 0.03084951,
                "r": 0.00578543,
                "i": 0.0015424,
                "z": 0.00043015,
                "y": 0.00026005,
            }
        else:
            self.dcr2_magnitudes = dcr2_magnitudes


        self.zd_col = "zenithDistance"
        self.pa_col = "PA"
        self.filter_col = filter_col
        self.ra_col = ra_col
        self.dec_col = dec_col
        self.degrees = degrees
        self.cols_req = [filter_col, ra_col, dec_col, alt_col, lst_col]
        #  'zenithDist', 'PA', 'HA' are additional columns required, coming
        #  from other stackers which must also be configured -- so we handle
        #  this explicitly here.
        self.zstacker = ZenithDistStacker(alt_col=alt_col, degrees=self.degrees)
        self.pastacker = ParallacticAngleStacker(
            ra_col=ra_col,
            dec_col=dec_col,
            mjd_col=mjd_col,
            degrees=self.degrees,
            lst_col=lst_col,
            site=site,
        )
        self.ra_and_dec = ra_and_dec
        # Note that RA/Dec could be coming from a dither stacker!
        # But we will assume that coord stackers will be handled separately.

    def _run(self, sim_data, cols_present=False):
        if cols_present:
            # Column already present in data; assume it is correct and does not
            # need recalculating.
            return sim_data
            
        # Need to make sure the Zenith stacker gets run first Call _run method
        # because already added these columns due to 'colsAdded' line.
        sim_data = self.zstacker.run(sim_data)
        sim_data = self.pastacker.run(sim_data)

        # if self.degrees:
        #     zenith_ang = sim_data[self.zd_col]
        #     # parallactic_angle = np.radians(sim_data[self.pa_col])
        # else:
        #     zenith_ang = np.rad2deg(sim_data[self.zd_col])
        #     # parallactic_angle = sim_data[self.pa_col]
        
        
        if self.degrees:
            zenith_tan = np.tan(np.radians(sim_data[self.zd_col]))
            parallactic_angle = np.radians(sim_data[self.pa_col])
        else:
            zenith_tan = np.tan(sim_data[self.zd_col])
            parallactic_angle = sim_data[self.pa_col]

        dcr_var = zenith_tan**2
        dcr_moment_rara = dcr_var * np.sin(parallactic_angle)**2 # shape moment along ra
        dcr_moment_decdec = dcr_var * np.cos(parallactic_angle)**2 # shape moment along dec
        dcr_moment_radec = dcr_var * np.sin(parallactic_angle) * np.cos(parallactic_angle) #covariance moment between ra and dec
        
        for filtername in np.unique(sim_data[self.filter_col]):
            fmatch = np.where(sim_data[self.filter_col] == filtername)
            dcr_var[fmatch] = self.dcr2_magnitudes[filtername] * dcr_var[fmatch]

            if self.ra_and_dec:
                dcr_moment_rara[fmatch] = self.dcr2_magnitudes[filtername] * dcr_moment_rara[fmatch] 
                dcr_moment_decdec[fmatch] = self.dcr2_magnitudes[filtername] * dcr_moment_decdec[fmatch] 
                dcr_moment_radec[fmatch] = self.dcr2_magnitudes[filtername] * dcr_moment_radec[fmatch] 
            
        sim_data["dcr_var"] = dcr_var
        
        if self.ra_and_dec:
            sim_data["ra_dcr_var"] = dcr_moment_rara
            sim_data["dec_dcr_var"] = dcr_moment_decdec
            sim_data["ra_dec_dcr_cov"] = dcr_moment_radec
            
        return sim_data
    


## Test the stacker:

In [9]:
# --- Testing the DcrStacker, run the stacker and print the first few entries

from rubin_sim.maf.stackers import DcrStacker
from rubin_sim.maf.utils import get_sim_data

nside = 128

# --- Get the data and run the original DCR stacker (This can take a while)
opsim_fname = get_baseline()
db = get_sim_data(opsim_fname)
stacker = DcrStacker()
obs_stacked = stacker.run(db)



In [15]:
# --- Run the new stacker
dcr2_stacker = Dcr2Stacker(ra_and_dec = False)
obs_stacked_2 = dcr2_stacker.run(obs_stacked)

/opt/miniconda3/envs/DESC_DCR/lib/python3.13/site-packages/rubin_sim/maf/stackers/base_stacker.py:139: UserWarning: Warning - column zenithDistance already present in sim_data, may be overwritten (depending on stacker).
  warnings.warn(
/opt/miniconda3/envs/DESC_DCR/lib/python3.13/site-packages/rubin_sim/maf/stackers/base_stacker.py:139: UserWarning: Warning - column PA already present in sim_data, may be overwritten (depending on stacker).
  warnings.warn(


In [16]:
# --- Print a few rows of the data to check the new columns 

# --- create mask to view only a subset of data
mask = np.abs(obs_stacked['fieldDec'] + 90) < 0.5
obs_stacked_filtered = obs_stacked_2[mask]

# AltAzStacker adds 'altitude' and 'azimuth'
# DCRStacker adds 'ra_dcr_amp' and 'dec_dcr_amp'
# DCR2Stacker adds 'dcr_var', 'ra_dcr_var', 'dec_dcr_var', and 'ra_dec_dcr_cov' if ra_and_dec = True
# ParallacticAngleStacker adds 'PA'
# cols_to_print = ['filter', 'fieldRA', 'fieldDec', 'altitude', 'azimuth', 'PA', "ra_dcr_amp", "dec_dcr_amp", 'dcr_var', "ra_dcr_var", "dec_dcr_var", "ra_dec_dcr_cov"]
cols_to_print = ['filter', 'fieldRA', 'fieldDec', 'altitude', 'PA', 'dcr_var', "ra_dcr_var", "dec_dcr_var", "ra_dec_dcr_cov"]
print(cols_to_print)
print(np.array(obs_stacked_filtered[cols_to_print][:10]))

print(cols_to_print)
print(np.array(obs_stacked_2[cols_to_print][:10]))

['filter', 'fieldRA', 'fieldDec', 'altitude', 'PA', 'dcr_var', 'ra_dcr_var', 'dec_dcr_var', 'ra_dec_dcr_cov']
[('r', 140.98318568, -89.67761069, 30.07406248, -166.6302765 , 0.01725305, 0., 0., 0.)
 ('z', 120.04456997, -89.58198357, 30.06681663,  103.48523852, 0.00128352, 0., 0., 0.)
 ('y', 120.04456997, -89.58198357, 30.0449852 ,  107.67154951, 0.00077733, 0., 0., 0.)
 ('u', 251.8091071 , -89.77358924, 30.03915746, -116.978259  , 0.07192204, 0., 0., 0.)
 ('g', 251.8091071 , -89.77358924, 30.05899704, -106.26456884, 0.09210968, 0., 0., 0.)
 ('g',  12.72060841, -89.56027674, 30.7777008 ,   30.40302893, 0.08696614, 0., 0., 0.)
 ('r',  12.72060841, -89.56027674, 30.71944086,   39.11452312, 0.01638504, 0., 0., 0.)
 ('r',  12.72060841, -89.56027674, 30.47299506,   66.75998791, 0.01670995, 0., 0., 0.)]
['filter', 'fieldRA', 'fieldDec', 'altitude', 'PA', 'dcr_var', 'ra_dcr_var', 'dec_dcr_var', 'ra_dec_dcr_cov']
[('r', 262.8511067 , -22.59115878, 30.26556459, 112.95608359, 0.01698964, 0., 0., 0